# Лабораторная работа 01. Первичный анализ данных с NumPy

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

**Задача аналитика:** получить таблицу автомобилей, разобраться в её устройстве, описать цены и пробеги и найти значения, которые нужно проверить перед дальнейшей работой.

**Цель:** освоить массивы, индексацию, поэлементные операции и базовые статистики NumPy на реальных табличных данных; понять различие между отдельным наблюдением, распределением значений и статистикой выборки.

Один автомобиль отличается от другого: цена и пробег не имеют единственного значения для всей таблицы. Мы будем изучать этот разброс. **Реальные данные не обязательно «случайный шум».** Вариация связана с характеристиками объектов, составом набора, условиями наблюдения и ошибками записи.

После работы вы сможете загрузить числовые столбцы CSV, выбрать записи по условию, сравнить среднее с медианой, описать разброс и написать вывод с оговорками.

**Порядок:** выполняйте готовые ячейки сверху вниз. Отвечайте на вопросы под ними. В конце выполните самостоятельное мини-исследование. Прогноз цены, Pandas и построение графиков здесь не нужны.

## 1. От вопроса к таблице

Мы продолжаем сквозную задачу курса — данные об автомобилях на вторичном рынке. В этой работе изучаем **содержимое предоставленного файла**, а не делаем выводы обо всём рынке.

До расчётов аналитик уточняет:

- что является наблюдением: автомобиль, объявление или совершённая сделка;
- где и когда собраны данные;
- является ли цена указанной в объявлении или фактической ценой сделки;
- как измерен пробег и что означает пропуск.

Имена столбцов сами по себе не отвечают на эти вопросы. Не называйте сумму указанных цен выручкой без сведений о сделках. Не подставляйте сегодняшнюю дату как дату сбора старой таблицы.

| Столбец CSV | Содержание | Представление |
|---|---|---|
| Make, Model | Марка и модель | Категории / текст |
| Year | Год выпуска | Числовой признак, целые значения |
| Distance | Указанный пробег | Числовой признак, км |
| Engine_capacity(cm3) | Объём двигателя | Числовой признак, см³ |
| Fuel_type | Тип топлива | Категория |
| AutoTransmission | Признак автоматической коробки | Категория True/False |
| Price(euro) | Указанная цена | Числовой признак, евро |

**Строка** — одна запись. **Столбец** — один признак. **Вектор признака** — значения этого признака для всех строк. Порядок строк связывает признаки одного автомобиля.

### Импортируем NumPy

NumPy выполняет численные операции над массивами. В этой работе не используем Pandas. Библиотека `csv` потребуется только для просмотра заголовка исходного файла.

In [ ]:
from pathlib import Path
import csv
import urllib.request
import numpy as np

np.set_printoptions(suppress=True, precision=2)
print('NumPy:', np.__version__)

**Вопрос:** Какие четыре числовых признака будем читать? Укажите единицу каждого из них.

*Ваш ответ:*

### Получаем файл один раз

Положите `cars.csv` рядом с ноутбуком. Если его нет, готовая ячейка скачает файл курса. В дальнейшем читаем локальную копию. CSV — текстовый файл: значения разделены запятыми, первая строка содержит названия столбцов.

In [ ]:
data_path = Path('cars.csv')
URL = 'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/refs/heads/main/Workshops/cars.csv'
if not data_path.exists():
    urllib.request.urlretrieve(URL, data_path)

with data_path.open(encoding='utf-8-sig', newline='') as file:
    reader = csv.reader(file)
    header = next(reader)
    first_rows = [next(reader) for _ in range(3)]
print('Столбцы:', header)
for row in first_rows:
    print(row)

**Вопрос:** Чем заголовок отличается от первой записи? Можем ли мы по трём строкам описать весь набор?

*Ваш ответ:*

### Читаем числовые столбцы

`genfromtxt` умеет прочитать числа из текстовой таблицы. `delimiter` задаёт разделитель, `skip_header=1` пропускает заголовок, `usecols` выбирает номера столбцов. Нумерация начинается с нуля. Для данного файла числовые поля не содержат CSV-разделителя внутри значения; сложные CSV будем разбирать в работе с Pandas.

In [ ]:
numeric_names = ['Year', 'Distance', 'Engine_capacity(cm3)', 'Price(euro)']
numeric_columns = [header.index(name) for name in numeric_names]
X = np.genfromtxt(data_path, delimiter=',', skip_header=1,
                  usecols=numeric_columns, dtype=float, ndmin=2)
print('Первые пять записей:')
print(X[:5])
print('Порядок столбцов:', numeric_names)

**Вопрос:** Почему в этой матрице нет марки и модели? Что означает число в позиции X[0, 3]?

*Ваш ответ:*

## 2. Осваиваем массив на загруженных данных

`numpy.ndarray` — массив с общим типом элементов (`dtype`). Он может иметь одну или несколько осей. В нашей двумерной матрице ось 0 перебирает записи, ось 1 — признаки.

NumPy поддерживает не только числа, но и другие типы. Для числовых операций здесь используем `float`. Даже целый по смыслу год представлен как `2011.0`: числовая матрица имеет общий тип, а пропуски удобно представить через `nan`.

Порядок элементов и форма имеют значение. Не сортируйте один столбец отдельно, если хотите сохранить соответствие характеристик автомобиля.

### Форма, размер и тип

`shape` — размеры по осям, `ndim` — число осей, `size` — число всех элементов, `dtype` — тип элемента. Не путайте число автомобилей и число ячеек матрицы.

In [ ]:
print('shape:', X.shape)
print('ndim :', X.ndim)
print('size :', X.size)
print('dtype:', X.dtype)
n_rows, n_features = X.shape
row_id = np.arange(n_rows)
print('Записей:', n_rows, 'признаков:', n_features)
print('Первые номера записей:', row_id[:5])

**Вопрос:** Почему X.size в четыре раза больше числа записей? Что означает shape=(n,) у вектора?

*Ваш ответ:*

### Индексация и срезы

Запись `X[строка, столбец]` выбирает значение. Двоеточие означает выбор всех элементов по оси. Срез `:5` берёт позиции 0–4, исключая 5.

In [ ]:
year = X[:, 0]
distance = X[:, 1]
engine = X[:, 2]
price = X[:, 3]

print('Форма вектора цены:', price.shape)
print('Первая цена:', price[0])
print('Последняя цена:', price[-1])
print('Пять первых цен:', price[:5])
print('Записи 10–14:')
print(X[10:15, :])
print('Запись 100:', X[100, :])

**Вопрос:** Как получить пробег записи 100? Сколько элементов в X[10:15, 3]?

*Ваш ответ:*

### Почему массив отличается от списка

Список Python удобен для хранения объектов. Умножение списка на 2 повторяет его элементы. Умножение числового массива на 2 изменяет каждое значение. Сравним на одних данных.

In [ ]:
five_prices = price[:5].copy()
price_list = five_prices.tolist()
print('Список * 2:', price_list * 2)
print('Массив * 2:', five_prices * 2)
print('Массив + 100:', five_prices + 100)
print('Созданный массив:', np.array(price_list))

**Вопрос:** Сколько элементов получилось в каждом случае? Изменилась ли исходная цена в файле?

*Ваш ответ:*

### Переводим единицы без цикла

Одна операция применяется ко всем элементам. Это векторизация. Перевод единиц должен быть обоснован смыслом признака: литры удобны для описания двигателя, а курс валют без даты вводить не будем.

In [ ]:
engine_litres = engine / 1000.0
distance_thousands = distance / 1000.0
print('Двигатель, литры:', engine_litres[:5])
print('Пробег, тысячи км:', distance_thousands[:5])
print('Форма сохранилась:', engine_litres.shape == engine.shape)

**Вопрос:** Напишите в новой ячейке перевод distance в мили: 1 миля = 1.609 км. Почему результат остаётся вектором?

*Ваш ответ:*

### Осторожно: срез может ссылаться на исходные данные

`price = X[:, 3]` обычно создаёт представление столбца, а не независимую копию. Изменение такого вектора может изменить X. Для самостоятельного изменения используйте `.copy()`. В этой работе сохраняем исходную матрицу.

Для поэлементного сравнения удобно иметь одинаковую форму массивов. Но NumPy допускает **broadcasting** совместимых форм. Поэтому утверждение «любые операции требуют одинаковых размеров» неверно. Основной маршрут использует простые пары одинаковых векторов; пример broadcasting находится в дополнительной части.

## 3. Сначала проверяем пригодность значений

Пропуск, подозрительное значение и редкий реальный автомобиль — разные случаи. Аналитик сначала замечает и описывает их. Он не объявляет все крайние значения ошибочными и не удаляет их автоматически.

`np.nan` — специальное числовое обозначение отсутствующего или не прочитанного значения. Бесконечность также нельзя включать в обычное описание цен.

### Находим пропуски и бесконечности

`isnan` ищет nan. `isfinite` возвращает True для обычных конечных чисел. Эти функции дают логический массив той же формы, что и вход.

In [ ]:
finite = np.isfinite(X)
missing_count = np.isnan(X).sum(axis=0)
nonfinite_count = (~finite).sum(axis=0)
for name, missing, nonfinite in zip(numeric_names, missing_count, nonfinite_count):
    print(f'{name}: nan={missing}, всех неконечных={nonfinite}')
print('Средний engine обычным mean:', np.mean(engine))
print('Средний engine через nanmean:', np.nanmean(engine))

**Вопрос:** Почему nanmean — не полноценная очистка? Обработает ли она бесконечность или нулевой объём двигателя?

*Ваш ответ:*

### Строим логическую маску

Сравнение каждого числа с порогом возвращает True/False. Маска выбирает строки. Считаем количество и долю дорогих записей, явно указывая знаменатель.

In [ ]:
known_price = np.isfinite(price)
price_values = price[known_price]
price_rows = row_id[known_price]
if price_values.size == 0:
    raise ValueError('Нет конечных значений цены для анализа.')
expensive = price_values > 10_000
print('Конечных цен:', price_values.size, 'из', n_rows)
print('Цен выше 10 000 евро:', np.count_nonzero(expensive))
print('Доля среди конечных цен:', np.mean(expensive))
print('Примеры цен:', price_values[expensive][:5])

**Вопрос:** Почему np.mean логической маски даёт долю? Как изменится вопрос при пороге 20 000 евро?

*Ваш ответ:*

### Объединяем условия

Используем `&` для «и», `|` для «или»; каждое условие заключаем в скобки. Ниже пороги служат поводом проверить запись, а не доказательством ошибки.

In [ ]:
low_distance_old_car = np.isfinite(distance) & np.isfinite(year) & (distance < 1000) & (year < 2020)
extreme_distance = np.isfinite(distance) & (distance > 1_000_000)
nonpositive_engine = np.isfinite(engine) & (engine <= 0)
print('Малый пробег при раннем годе выпуска:', low_distance_old_car.sum())
print('Пробег выше 1 млн км:', extreme_distance.sum())
print('Неположительный объём:', nonpositive_engine.sum())
print('Первые записи для проверки [номер, год, пробег, двигатель, цена]:')
print(np.column_stack((row_id[low_distance_old_car], X[low_distance_old_car]))[:5])

**Вопрос:** Можно ли автоматически удалить старый автомобиль с маленьким пробегом? Какие объяснения, кроме ошибки, возможны?

*Ваш ответ:*

В следующих статистиках используем все **конечные** значения выбранного признака. Мы пока не исправляем подозрительные записи и не применяем пороги очистки. Поэтому итоги описывают исходный файл и могут сильно зависеть от качества записей.

Для разных признаков число доступных значений может отличаться. При сравнении двух признаков одного автомобиля нужна общая маска по обоим столбцам; иначе можно потерять соответствие строк.

## 4. Описываем цены: типичный уровень и разброс

Один автомобиль — одно наблюдение цены. Столбец — множество наблюдений. **Статистика** — число, которое мы вычисляем по этому множеству.

Если рассматриваем случайно выбранную запись, её цена заранее неизвестна: удобно говорить о случайной величине и её распределении. Значения в уже сохранённом файле при этом фиксированы. Случайный выбор строк не превращает сам файл в репрезентативную выборку всего рынка.

### Среднее и медиана

Среднее — сумма, делённая на количество. Медиана разделяет упорядоченные значения пополам. Оба числа описывают уровень, но чувствительны к разным особенностям набора.

In [ ]:
mean_price = np.mean(price_values)
median_price = np.median(price_values)
print(f'Среднее: {mean_price:.2f} евро')
print(f'Медиана: {median_price:.2f} евро')
print(f'Минимум: {np.min(price_values):.2f} евро')
print(f'Максимум: {np.max(price_values):.2f} евро')

**Вопрос:** Какую статистику вы бы использовали в фразе «типичная цена в этом файле»? Почему одного числа недостаточно?

*Ваш ответ:*

### Находим записи с крайними значениями

argmax возвращает позицию внутри вектора. После фильтрации эта позиция может отличаться от номера в исходной матрице. Сохраняем соответствие через price_rows.

In [ ]:
position_max = np.argmax(price_values)
original_row_max = price_rows[position_max]
print('Позиция в векторе конечных цен:', position_max)
print('Номер исходной записи:', original_row_max)
print('Признаки этой записи:', X[original_row_max])

top_positions = np.argsort(price_values)[-5:][::-1]
print('Пять самых высоких цен: [номер, цена]')
print(np.column_stack((price_rows[top_positions], price_values[top_positions])))

**Вопрос:** Может ли максимум быть реальной ценой? Что нужно проверить до утверждения «это ошибка»?

*Ваш ответ:*

### Понимаем чувствительность среднего

Создадим небольшой учебный набор и изменим одно наблюдение. Это отдельный эксперимент, а не изменение исходной таблицы.

In [ ]:
demo_prices = np.array([4000., 5000., 6000., 7000., 8000.])
demo_with_extreme = demo_prices.copy()
demo_with_extreme[-1] = 100_000.
for label, values in [('Исходный пример', demo_prices), ('С крайним значением', demo_with_extreme)]:
    print(label, 'среднее=', np.mean(values), 'медиана=', np.median(values))

**Вопрос:** Какая статистика изменилась сильнее? Почему это полезно при изучении исходной таблицы?

*Ваш ответ:*

### Квантили: описываем несколько частей распределения

Квантиль уровня q показывает границу, соответствующую доле q упорядоченных наблюдений. Q1 и Q3 — уровни 0.25 и 0.75; между ними лежит центральная половина распределения. При повторяющихся значениях и интерполяции доли не обязаны совпадать точно.

In [ ]:
levels = np.array([0.10, 0.25, 0.50, 0.75, 0.90])
quantiles = np.quantile(price_values, levels)
for level, value in zip(levels, quantiles):
    print(f'q={level:.2f}: {value:.2f} евро')
q1, q3 = np.quantile(price_values, [0.25, 0.75])
iqr = q3 - q1
print(f'IQR: {iqr:.2f} евро')

**Вопрос:** Как объяснить q=0.90 человеку без Python? Чем IQR отличается от разности максимума и минимума?

*Ваш ответ:*

### Стандартное отклонение и единицы

Стандартное отклонение описывает масштаб разброса относительно среднего. Оно чувствительно к крайним значениям. Используем ddof=0 для описания данного набора; оценивание дисперсии генеральной совокупности требует дополнительных предположений.

In [ ]:
std_price = np.std(price_values, ddof=0)
variance_price = np.var(price_values, ddof=0)
print(f'Стандартное отклонение: {std_price:.2f} евро')
print(f'Дисперсия: {variance_price:.2f} евро²')
print(f'IQR: {iqr:.2f} евро')

**Вопрос:** Почему у дисперсии единица евро²? Можно ли по одному стандартному отклонению заключить, что распределение нормальное?

*Ваш ответ:*

### Частоты по ценовым интервалам

Распределение можно описать частотами без графика. `histogram` здесь возвращает числа записей по заданным интервалам. Большинство интервалов включает левую границу и исключает правую; последний включает правую.

In [ ]:
bins = np.array([0., 5000., 10_000., 20_000., 50_000., np.inf])
nonnegative_prices = price_values[price_values >= 0]
counts, edges = np.histogram(nonnegative_prices, bins=bins)
print('Число неотрицательных конечных цен:', nonnegative_prices.size)
print('Отрицательных конечных цен:', np.count_nonzero(price_values < 0))
for left, right, count in zip(edges[:-1], edges[1:], counts):
    share = count / nonnegative_prices.size if nonnegative_prices.size else np.nan
    print(f'[{left:g}, {right:g}): {count} записей, доля={share:.3f}')

**Вопрос:** В каком интервале больше всего записей? Совпадает ли этот интервал со средней ценой?

*Ваш ответ:*

**Промежуточный вывод:** опишите цены в трёх предложениях. Укажите типичный уровень, разброс и записи, которые требуют проверки. Слова «много», «дорого», «большой разброс» подкрепляйте числами и единицами. Не называйте крайнее значение ошибкой только по его величине.

*Ваш вывод:*

## 5. Сравниваем части таблицы

Аналитик часто сравнивает группы, а не только все записи вместе. Введём учебные группы по году выпуска. Граница 2015 выбрана для примера; это не закон рынка. Сначала считаем размер каждой группы, затем её статистики.

### Разные группы — разные распределения

Общая маска сохраняет только строки с конечным годом и ценой. Сравнение групп описательное: оно само по себе не устанавливает причинное влияние года на цену.

In [ ]:
known_year_price = np.isfinite(year) & np.isfinite(price)
older = known_year_price & (year < 2015)
newer = known_year_price & (year >= 2015)
for label, mask in [('До 2015', older), ('2015 и позже', newer)]:
    group_prices = price[mask]
    print(label, 'n=', group_prices.size)
    if group_prices.size:
        print('  среднее=', round(float(np.mean(group_prices)), 2),
              'медиана=', round(float(np.median(group_prices)), 2))
        print('  Q1, Q3=', np.quantile(group_prices, [0.25, 0.75]))

**Вопрос:** Какие различия между автомобилями, кроме года, могут влиять на сравнение этих групп?

*Ваш ответ:*

Не делайте вывод «год объясняет всю разницу». В группах могут различаться марки, типы двигателя, пробег и состав объявлений. На этом этапе мы описали **связь в имеющихся данных**, а не доказали причинность.

## 6. Почему статистика меняется от выборки к выборке

Полный файл — наш доступный набор. Из него можно случайно выбрать часть строк без повторений и посчитать те же статистики. При другом выборе строк получится другое число.

Это простой опыт с **выборочной изменчивостью**. Он не доказывает, что исходный файл является независимой случайной выборкой рынка. Никаких предположений о нормальном распределении цен мы не вводим.

### Первые строки и случайно выбранные строки

`default_rng` создаёт генератор случайных чисел. Seed делает повторяемым этот конкретный опыт. Выбираем позиции в векторе конечных цен; исходные цены не заменяются сгенерированными.

In [ ]:
rng = np.random.default_rng(42)
sample_size = min(100, price_values.size)
random_positions = rng.choice(price_values.size, size=sample_size, replace=False)
random_prices = price_values[random_positions]
first_prices = price_values[:sample_size]
print('Размер частей:', sample_size)
print('Весь файл: среднее=', np.mean(price_values), 'медиана=', np.median(price_values))
print('Первые строки: среднее=', np.mean(first_prices), 'медиана=', np.median(first_prices))
print('Случайные строки: среднее=', np.mean(random_prices), 'медиана=', np.median(random_prices))

**Вопрос:** Можно ли считать первые 100 строк случайной выборкой? Гарантирует ли случайный выбор 100 строк точное совпадение статистик?

*Ваш ответ:*

### Повторяем выбор несколько раз

Медленно повторим эксперимент пять раз. После этого соберём результаты в массив. Простой цикл нужен только для повторения опыта; статистики каждого набора считает NumPy.

In [ ]:
sample_means = []
sample_medians = []
for experiment in range(5):
    positions = rng.choice(price_values.size, size=sample_size, replace=False)
    selected = price_values[positions]
    sample_means.append(np.mean(selected))
    sample_medians.append(np.median(selected))
sample_means = np.array(sample_means)
sample_medians = np.array(sample_medians)
print('Средние пяти частей:', sample_means)
print('Медианы пяти частей:', sample_medians)
print('Размах средних в этом опыте:', np.ptp(sample_means))

**Вопрос:** Почему результаты различаются, хотя формула и полный файл не менялись? Означает ли это, что вычисления ошибочны?

*Ваш ответ:*

Пять повторений — иллюстрация, а не оценка доверительного интервала. В отдельном запуске редкая высокая цена может сильно изменить среднее. Увеличение числа строк обычно помогает стабилизировать оценки в случайной выборке, но не исправляет систематические ошибки источника и не гарантирует улучшение в каждом единичном опыте.

**Главная мысль:** вычисленная статистика зависит от того, какие наблюдения попали в расчёт. Для вывода нужно указывать набор данных, условия отбора и количество записей.

## 7. Самостоятельное мини-исследование

Используйте **пробег**. Не меняйте исходную матрицу. Для расчётов выберите конечные значения, а нулевые и очень большие значения отдельно отметьте как требующие интерпретации. Не удаляйте их молча.

### Задание 1. Размер и качество

Создайте маску `known_distance`, вектор `distance_values`. Выведите общее число записей, число конечных пробегов, число нулевых пробегов и число пробегов выше 1 млн км. Укажите, какие числа описывают пропуски, а какие — значения для проверки.

In [ ]:
# Ваш код: маска, выбор значений и подсчёт записей.

### Задание 2. Статистический портрет

Посчитайте среднее, медиану, минимум, максимум, стандартное отклонение, Q1, Q3 и IQR пробега. Укажите единицы. Объясните разницу между средним и медианой и то, почему максимум не описывает типичный автомобиль.

In [ ]:
# Ваш код: статистики пробега.

### Задание 3. Сравнение групп

Для автомобилей до 2015 года и с 2015 года включительно посчитайте количество доступных пробегов, медиану и Q1/Q3. Используйте совместную маску конечного года и пробега. Сформулируйте описательный вывод; не объявляйте причинность.

In [ ]:
# Ваш код: общая маска, группы и статистики.

### Задание 4. Выборочная изменчивость

Выберите случайно 100 конечных пробегов без повторений, затем повторите выбор пять раз. Сравните медианы с медианой всего доступного вектора. Задайте seed, чтобы опыт можно было повторить. Объясните, что именно случайно в этом опыте.

In [ ]:
# Ваш код: генератор, выбор строк и сравнение медиан.

### Отчёт аналитика

Напишите 5–7 предложений:

1. Какие записи и признаки изучены, сколько значений вошло в расчёты.
2. Каков типичный пробег и насколько он различается между автомобилями.
3. Что изменилось при сравнении двух групп.
4. Какие значения требуют проверки и почему нельзя автоматически назвать их ошибками.
5. Чем отличаются статистики частей таблицы и какие ограничения есть у вывода.

*Ваш отчёт:*

## 8. Итоги и переход к следующей работе

В 01 мы загрузили числовые столбцы, освоили массив и описали исходные данные. В следующей работе Pandas и ETL понадобятся для более полной подготовки таблицы: работы с именами и категориями, исправления типов, обработки пропусков и сохранения подготовленного набора.

В этой работе не нужно обучать модель цены, подбирать коэффициенты или заменять первичный анализ сложными графиками.

| Критерий | Баллы |
|---|---:|
| Загрузка, форма, индексация и единицы | 2 |
| Маски, количества и качество значений | 2 |
| Статистики и их интерпретация | 3 |
| Группы и выборочная изменчивость | 2 |
| Краткий вывод с ограничениями | 1 |
| Итого | 10 |

**Контрольные вопросы:**

- Чем ndarray отличается от списка? Что означают shape, dtype и axis?
- Что является наблюдением, признаком и статистикой в нашей работе?
- Чем пропуск отличается от редкого значения?
- Почему среднее и медиана могут сильно различаться?
- Что показывает IQR, а что стандартное отклонение?
- Почему случайная часть файла не гарантирует представительность рынка?
- Почему при сравнении двух признаков нельзя независимо удалить строки в каждом столбце?

## Дополнительно. Оси и broadcasting

Эта часть закрепляет NumPy, если основная работа уже выполнена. В исходной матрице не надо усреднять год, цену и пробег внутри одной строки: единицы различаются. Поэтому для примера возьмём цены трёх автомобилей из двух условных источников — все ячейки имеют одну единицу.

In [ ]:
two_sources = np.array([[5000., 5200.], [8000., 8400.], [10_000., 9800.]])
print('Матрица: автомобили × источники')
print(two_sources)
print('Среднее по автомобилям для каждого источника, axis=0:', two_sources.mean(axis=0))
print('Среднее по источникам для каждого автомобиля, axis=1:', two_sources.mean(axis=1))

three_prices = np.array([5000., 8000., 10_000.])
column = three_prices.reshape(-1, 1)
print('Форма вектора:', three_prices.shape, 'форма столбца:', column.shape)
print('Столбец + вектор: форма', (column + three_prices).shape)
print(column + three_prices)
print('Столбец после транспонирования:', column.T.shape)

**Вопрос:** почему сложение массива формы (3, 1) и массива формы (3,) даёт матрицу (3, 3)? Почему одномерный вектор при `.T` сам по себе не становится столбцом?

**Справка:** `np.arange` создаёт значения с заданным шагом; `np.linspace` — заданное число равноотстоящих значений; `np.zeros` и `np.ones` — массивы нулей и единиц. Созданные таким способом числа не являются новыми наблюдениями рынка.

- [Руководство NumPy для начинающих](https://numpy.org/doc/stable/user/absolute_beginners.html).
- [Описание genfromtxt](https://numpy.org/doc/stable/reference/generated/numpy.genfromtxt.html).
- [Правила broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).